# The Price is Right - Simple Pricer

Predict product price from description using the week 6 dataset: load data from Hugging Face, run a **prompt-based baseline**, then **fine-tune** a small OpenAI model and evaluate with the shared tester.

**Flow:** Load data -> Baseline (API) -> Fine-tune -> Evaluate fine-tuned model. Optional: Hugging Face baseline.

## 1. Setup

Add the week6 folder to the path so we can use the shared `pricer` package. Load env and Hugging Face token (needed to load the dataset from the Hub).

In [ ]:
import os
import sys
import re
import json

# Use the week6 pricer package (run this notebook from the sammyloto folder)
week6_root = os.path.join(os.getcwd(), "..", "..")
sys.path.insert(0, week6_root)
from pricer.items import Item
from pricer.evaluator import evaluate

from dotenv import load_dotenv
from openai import OpenAI

# Load .env so HF_TOKEN and OPENAI_API_KEY are set (dataset loader uses HF_TOKEN automatically)
load_dotenv(override=True)

# Optional: call login() only if you need to validate the token (e.g. for gated models).
# For public datasets, HF_TOKEN in the environment is enough; skip login to avoid 401 whoami issues.
# from huggingface_hub import login
# if os.environ.get("HF_TOKEN"):
#     login(os.environ["HF_TOKEN"], add_to_git_credential=True)

openai = OpenAI()
print("Setup OK - pricer and OpenAI ready. HF_TOKEN from .env will be used when loading the dataset.")

## 2. Load data from Hugging Face

Use the shared dataset (replace `ed-donner` with your HF username if you pushed your own). Set `LITE_MODE = True` for a smaller dataset.

In [ ]:
LITE_MODE = True  # True = items_lite, False = items_full
USERNAME = "ed-donner"
dataset_name = f"{USERNAME}/items_lite" if LITE_MODE else f"{USERNAME}/items_full"

train, val, test = Item.from_hub(dataset_name)

# Ensure every item has a prompt (for test_prompt() later)
for item in train + val + test:
    if not item.prompt and (item.summary or item.title):
        item.make_prompt((item.summary or item.title))

print(f"Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")

## 3. Baseline: prompt-based API predictor

One simple prompt: ask the model to estimate price and reply with only the number. The evaluator parses the reply (handles "$", commas, decimals).

In [ ]:
BASELINE_MODEL = "gpt-4o-mini"  # or gpt-4.1-nano-2025-04-14

def baseline_pricer(item):
    """Call OpenAI with a simple instruction; return the model's reply (evaluator will parse the number)."""
    user_msg = f"Estimate the price in USD for this product. Reply with only the price, no explanation.\n\n{item.summary or item.title}"
    r = openai.chat.completions.create(
        model=BASELINE_MODEL,
        messages=[{"role": "user", "content": user_msg}],
        max_tokens=15,
    )
    return r.choices[0].message.content or "0"

In [ ]:
# Run the shared evaluator (default 200 test points)
evaluate(baseline_pricer, test)

## 4. Fine-tune a small model

Use a small subset of training data (OpenAI recommends 50–100+ examples). Build JSONL with user/assistant messages, upload files, start a fine-tuning job, then use the resulting model as the predictor.

In [ ]:
# Fine-tune config: small = fast and cheap
FT_TRAIN_SIZE = 100
FT_VAL_SIZE = 50
fine_tuned_model_name = None  # set later when the fine-tuning job has succeeded

ft_train = train[:FT_TRAIN_SIZE]
ft_val = val[:FT_VAL_SIZE]

In [ ]:
def messages_for_finetune(item):
    """One training example: user = product text, assistant = price."""
    text = item.summary or item.title
    user = f"Estimate the price of this product. Respond with the price, no explanation.\n\n{text}"
    assistant = f"${item.price:.2f}"
    return [{"role": "user", "content": user}, {"role": "assistant", "content": assistant}]

def make_jsonl(items):
    out = []
    for item in items:
        out.append({"messages": messages_for_finetune(item)})
    return "\n".join(json.dumps(obj) for obj in out)

In [ ]:
os.makedirs("jsonl", exist_ok=True)
with open("jsonl/ft_train.jsonl", "w") as f:
    f.write(make_jsonl(ft_train))
with open("jsonl/ft_val.jsonl", "w") as f:
    f.write(make_jsonl(ft_val))
print("Wrote jsonl/ft_train.jsonl and jsonl/ft_val.jsonl")

In [ ]:
# Upload to OpenAI and create fine-tuning job
with open("jsonl/ft_train.jsonl", "rb") as f:
    train_file = openai.files.create(file=f, purpose="fine-tune")
with open("jsonl/ft_val.jsonl", "rb") as f:
    val_file = openai.files.create(file=f, purpose="fine-tune")

job = openai.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=val_file.id,
    model="gpt-4.1-nano-2025-04-14",
    seed=42,
    hyperparameters={"n_epochs": 1, "batch_size": 1},
    suffix="pricer",
)
# So later cells never raise NameError; next cell will overwrite when job has succeeded
fine_tuned_model_name = None
print(f"Job created: {job.id}. Wait until status is 'succeeded' on the OpenAI dashboard.")

In [ ]:
# After the job completes, get the model name (run this cell once the job status is "succeeded")
jobs = openai.fine_tuning.jobs.list(limit=1)
if jobs.data:
    job_id = jobs.data[0].id
    fine_tuned_model_name = openai.fine_tuning.jobs.retrieve(job_id).fine_tuned_model
    print(f"Using model: {fine_tuned_model_name}")
else:
    fine_tuned_model_name = None
    print("No fine-tuning job found. Run the previous cell first and wait for completion.")

In [ ]:
def test_messages_for(item):
    text = item.summary or item.title
    return [{"role": "user", "content": f"Estimate the price of this product. Respond with the price, no explanation.\n\n{text}"}]

def fine_tuned_pricer(item):
    if not fine_tuned_model_name:
        return "0"
    r = openai.chat.completions.create(
        model=fine_tuned_model_name,
        messages=test_messages_for(item),
        max_tokens=15,
    )
    return r.choices[0].message.content or "0"

In [ ]:
# Evaluate fine-tuned model (run after job has succeeded)
if fine_tuned_model_name:
    evaluate(fine_tuned_pricer, test)
else:
    print("Set fine_tuned_model_name first.")

## 5. Optional: Hugging Face baseline

A **free, local** option: use a small Hugging Face model to predict price from the product text. Useful if you want to avoid API calls or compare with an open model.

**What you need:**
1. **Install:** `pip install transformers torch` (and `accelerate` if you use GPU).
2. **Token:** Create a Hugging Face account at [huggingface.co](https://huggingface.co), then create a token in Settings → Access Tokens. Put it in `.env` as `HF_TOKEN=` (already used above for loading the dataset).
3. **Model:** The example uses `google/flan-t5-small`. It’s small and runs on CPU; for better quality you can try `google/flan-t5-base` (needs more RAM) or a small causal LM. First run may download the model.

**Note:** This baseline is usually worse than the OpenAI fine-tuned model; it’s here for learning and comparison.

In [ ]:
# Optional: uncomment to run the Hugging Face baseline
# %pip install -q transformers torch

# from transformers import pipeline
# import re

# HF_MODEL = "google/flan-t5-small"  # small and fast; or try "google/flan-t5-base"
# pipe = pipeline("text2text-generation", model=HF_MODEL, max_length=20)

# def parse_price_from_text(text):
#     """Extract a single number from model output (handles '50', '$50', '50.00', etc.)."""
#     text = str(text).strip().replace("$", "").replace(",", "")
#     match = re.search(r"[-+]?\\d*\\.?\\d+", text)
#     return float(match.group()) if match else 0.0

# def hf_pricer(item):
#     prompt = f"Product: {item.summary or item.title}. Price in US dollars:"
#     out = pipe(prompt, max_new_tokens=10, do_sample=False)
#     return parse_price_from_text(out[0]["generated_text"])

# evaluate(hf_pricer, test, size=50)  # use size=50 for a quick check (HF can be slow)